# Flight Delay Prediction based on Weather Conditions (Spark Version)

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("FlightDelayWeatherPrediction") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

DATA_DIR = 'Data'
FLIGHTS_DIR = os.path.join(DATA_DIR, 'Flights')
WEATHER_DIR = os.path.join(DATA_DIR, 'Weather')
MAPPING_FILE = os.path.join(DATA_DIR, 'wban_airport_timezone.csv')

print(f"Flight files available: {len(os.listdir(FLIGHTS_DIR))}")
print(f"Weather files available: {len(os.listdir(WEATHER_DIR))}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 15:34:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 15:34:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/09 15:34:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Flight files available: 36
Weather files available: 8


### Step 1: Loading Airport-to-Weather Station Mapping
We link `ORIGIN_AIRPORT_ID` / `DEST_AIRPORT_ID` to their respective `WBAN` weather station codes.

In [2]:
df_mapping = spark.read.csv(MAPPING_FILE, header=True, inferSchema=True)
df_mapping_select = df_mapping.select(
    F.col("AirportID").cast("integer"), 
    F.col("WBAN").cast("integer")
)

print(f"Total mapped airports: {df_mapping_select.count()}")

Total mapped airports: 305


### Step 2: Data Loading & Preprocessing Functions
Functions to load and process data into PySpark DataFrames, managing schema types and robust timestamp parsing.

In [3]:
def load_flight_data(filepath, df_map):
    df_flights = spark.read.csv(filepath, header=True, inferSchema=True)
    
    # Select important columns and filter non-cancelled, non-diverted
    cols_to_keep = ['FL_DATE', 'ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID', 
                    'CRS_DEP_TIME', 'ARR_DELAY_NEW', 'CANCELLED', 'DIVERTED']
    df_flights = df_flights.select(*cols_to_keep) \
                           .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
    
    # Broadcast join to get WBAN mappings
    df_map_origin = df_map.withColumnRenamed("AirportID", "ORIGIN_AIRPORT_ID") \
                          .withColumnRenamed("WBAN", "ORIGIN_WBAN")
    df_map_dest = df_map.withColumnRenamed("AirportID", "DEST_AIRPORT_ID") \
                        .withColumnRenamed("WBAN", "DEST_WBAN")
                        
    df_flights = df_flights.join(F.broadcast(df_map_origin), on="ORIGIN_AIRPORT_ID", how="inner") \
                           .join(F.broadcast(df_map_dest), on="DEST_AIRPORT_ID", how="inner")
                           
    # Parse Scheduled Departure Time (combining FL_DATE and CRS_DEP_TIME)
    # CRS_DEP_TIME is like 1430 -> "1430"
    df_flights = df_flights.withColumn("DEP_TIME_STR", F.lpad(F.col("CRS_DEP_TIME").cast("string"), 4, "0"))
    
    df_flights = df_flights.withColumn(
        "SCHEDULED_DEP",
        F.expr("try_to_timestamp(concat_ws(' ', FL_DATE, DEP_TIME_STR), 'yyyy-MM-dd HHmm')")
    )
    df_flights = df_flights.filter(F.col("SCHEDULED_DEP").isNotNull())
    
    # Give a unique ID to each flight for later Window operations
    df_flights = df_flights.withColumn("FLIGHT_ID", F.monotonically_increasing_id())
    
    return df_flights.select("FLIGHT_ID", "SCHEDULED_DEP", "ORIGIN_WBAN", "DEST_WBAN", "ARR_DELAY_NEW")

def load_weather_data(filepath):
    df_weather = spark.read.csv(filepath, header=True, inferSchema=True)
    
    # Select cols
    cols_to_keep = ['WBAN', 'Date', 'Time', 'DryBulbCelsius', 'Visibility', 'WindSpeed']
    df_weather = df_weather.select(*cols_to_keep)
    
    # Cast weather conditions to numeric
    for col_name in ['DryBulbCelsius', 'Visibility', 'WindSpeed']:
        df_weather = df_weather.withColumn(col_name, F.expr(f"try_cast({col_name} as double)"))
        
    df_weather = df_weather.withColumn("Time_STR", F.lpad(F.col("Time").cast("string"), 4, "0"))
    df_weather = df_weather.withColumn(
        "OBS_TIME",
        F.expr("try_to_timestamp(concat_ws(' ', Date, Time_STR), 'yyyyMMdd HHmm')")
    )
    
    # Drop rows with missing timestamp and specific missing weather combinations
    df_weather = df_weather.filter(F.col("OBS_TIME").isNotNull())
    
    return df_weather.select("WBAN", "OBS_TIME", "DryBulbCelsius", "Visibility", "WindSpeed")

# Using the sample for January 2012
sample_flight_path = os.path.join(FLIGHTS_DIR, '201201.csv')
sample_weather_path = os.path.join(WEATHER_DIR, '201201hourly.txt')

print("Loading data into Spark DataFrames...")
df_flight_sample  = load_flight_data(sample_flight_path, df_mapping_select)
df_weather_sample = load_weather_data(sample_weather_path)

# Show schema and count
df_flight_sample.printSchema()
df_weather_sample.printSchema()

Loading data into Spark DataFrames...


root
 |-- FLIGHT_ID: long (nullable = false)
 |-- SCHEDULED_DEP: timestamp (nullable = true)
 |-- ORIGIN_WBAN: integer (nullable = true)
 |-- DEST_WBAN: integer (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)

root
 |-- WBAN: integer (nullable = true)
 |-- OBS_TIME: timestamp (nullable = true)
 |-- DryBulbCelsius: double (nullable = true)
 |-- Visibility: double (nullable = true)
 |-- WindSpeed: double (nullable = true)



### Step 3: DataFrame Caching
Because of the upcoming iterative table joins, we cache the datasets in memory.

In [4]:
df_flight_sample.cache()
df_weather_sample.cache()

print(f"Sample Flight Count: {df_flight_sample.count()}")
print(f"Sample Weather Count: {df_weather_sample.count()}")

Sample Flight Count: 453790


Sample Weather Count: 4192912


### Step 4: Building the 12-Hour Weather Window Features
Spark does not have `pd.merge_asof`. To mimic finding the *closest previous weather reading* for each hour minus `h` from 1 to 12 before the flight, we perform an interval join using a custom window approach. 

*In PySpark, we join on WBAN, limit observations up to 3 hours before the target time, and use `row_number` to grab the single closest observation.*

In [ ]:
def create_weather_features(df_flights, df_weather_all, max_hours=12):
    # Calculate the look-back window (up to max_hours before flight)
    df_window = df_flights.withColumn(
        "MIN_TIME", F.expr(f"SCHEDULED_DEP - INTERVAL {max_hours} HOURS")
    )
    
    # === ORIGIN WEATHER PROCESSING ===
    # Range Join for Origin
    df_org_join = df_window.join(
        df_weather_all,
        (df_window.ORIGIN_WBAN == df_weather_all.WBAN) & 
        (df_weather_all.OBS_TIME <= df_window.SCHEDULED_DEP) & 
        (df_weather_all.OBS_TIME >= df_window.MIN_TIME),
        "left"
    )
    
    # Calculate difference in hours (lag_h)
    df_org_lag = df_org_join.withColumn(
        "lag_h", 
        F.floor((F.unix_timestamp("SCHEDULED_DEP") - F.unix_timestamp("OBS_TIME")) / 3600)
    ).filter((F.col("lag_h") >= 0) & (F.col("lag_h") < max_hours))
    
    # Pivot logic for Origin
    df_org_pivot = df_org_lag.groupBy("FLIGHT_ID").pivot("lag_h", list(range(max_hours))).agg(
        F.avg("DryBulbCelsius").alias("DryBulbCelsius"),
        F.avg("Visibility").alias("Visibility"),
        F.avg("WindSpeed").alias("WindSpeed")
    )
    
    # Rename pivot columns for Origin
    org_renamed_cols = ["FLIGHT_ID"]
    for h in range(max_hours):
        for metric in ["DryBulbCelsius", "Visibility", "WindSpeed"]:
            old_col = f"{h}_{metric}"
            new_col = f"ORG_{metric}_H{h}"
            if old_col in df_org_pivot.columns: # Spark rename applies to existing columns
                df_org_pivot = df_org_pivot.withColumnRenamed(old_col, new_col)
                org_renamed_cols.append(new_col)
    
    # === DESTINATION WEATHER PROCESSING ===
    # Range Join for Destination
    df_dst_join = df_window.join(
        df_weather_all,
        (df_window.DEST_WBAN == df_weather_all.WBAN) & 
        (df_weather_all.OBS_TIME <= df_window.SCHEDULED_DEP) & 
        (df_weather_all.OBS_TIME >= df_window.MIN_TIME),
        "left"
    )
    
    # Calculate difference in hours (lag_h)
    df_dst_lag = df_dst_join.withColumn(
        "lag_h", 
        F.floor((F.unix_timestamp("SCHEDULED_DEP") - F.unix_timestamp("OBS_TIME")) / 3600)
    ).filter((F.col("lag_h") >= 0) & (F.col("lag_h") < max_hours))
    
    # Pivot logic for Destination
    df_dst_pivot = df_dst_lag.groupBy("FLIGHT_ID").pivot("lag_h", list(range(max_hours))).agg(
        F.avg("DryBulbCelsius").alias("DryBulbCelsius"),
        F.avg("Visibility").alias("Visibility"),
        F.avg("WindSpeed").alias("WindSpeed")
    )
    
    # Rename pivot columns for Destination
    dst_renamed_cols = ["FLIGHT_ID"]
    for h in range(max_hours):
        for metric in ["DryBulbCelsius", "Visibility", "WindSpeed"]:
            old_col = f"{h}_{metric}"
            new_col = f"DST_{metric}_H{h}"
            if old_col in df_dst_pivot.columns:
                df_dst_pivot = df_dst_pivot.withColumnRenamed(old_col, new_col)
                dst_renamed_cols.append(new_col)
                
    # === FINAL CLEANUP AND JOIN ===
    # Inner join combining flights and both feature pivots
    # Using a list of join keys ensures "FLIGHT_ID" is gracefully handled without duplication.
    df_final = df_flights.join(df_org_pivot, "FLIGHT_ID", "inner")                          .join(df_dst_pivot, "FLIGHT_ID", "inner")
                         
    # Drop rows without complete weather mappings
    df_final = df_final.dropna()
    
    return df_final

# To respect execution times, we'll limit the processing dataset, just like the pandas notebook (.head(50000))
df_flight_subset = df_flight_sample  # no .limit()

print("Starting weather feature extraction across Spark Range Join and Pivot Strategy...")
print("This scales perfectly compared to iterated left joins.")
df_final = create_weather_features(df_flight_subset, df_weather_sample, max_hours=12)

# Cache final prepared dataset for downstream modeling
df_final.cache()
print(f"Final Dataset Length: {df_final.count()}")
df_final.show(5, truncate=False)

Starting weather feature extraction across Spark Range Join and Pivot Strategy...
This scales perfectly compared to iterated left joins.


Final Dataset Length: 437363
+---------+-------------------+-----------+---------+-------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+------------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+----------------------+------------------+-----------------+----------------------+------------------+-----------------+---------------------+-----------------+-----------------+---------------------+-----------------+----------------+---------------------+-----------------+----------------+---------------------+-----------------+

### Step 5: Prediction with Spark ML Random Forest (Balanced)

We predict binary delays: 1 if delayed > 15 minutes, otherwise 0. To handle extreme class imbalance (vast majority of flights are on-time), we perform Random Under-Sampling to balance the dataset 50/50. 
We then train a highly parallelized `RandomForestClassifier`.

In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Feature columns: all origin and destination weather lags
feature_cols = [c for c in df_final.columns if c.startswith("ORG_") or c.startswith("DST_")]

# Create Label column
df_model = df_final.withColumn(
    "label", 
    F.when(F.col("ARR_DELAY_NEW") > 15, 1).otherwise(0)
)

print(f"Total Initial Samples: {df_model.count()} | Features: {len(feature_cols)}")

# --- RANDOM UNDER-SAMPLING ---
# Find counts
minority_count = df_model.filter(F.col("label") == 1).count()

# Ensure we don't under-sample completely if the minority count is extremely low or 0
if minority_count > 0:
    majority_df = df_model.filter(F.col("label") == 0).orderBy(F.rand(seed=42)).limit(minority_count)
    minority_df = df_model.filter(F.col("label") == 1)
    
    # Combine back 50/50 dataset
    df_model = majority_df.union(minority_df)
    print(f"Balanced Dataset Samples: {df_model.count()} (50% delayed, 50% on-time)")
else:
    print("Warning: No delays found in this dataset subset. Cannot perform undersampling.")

# Assemble features into a single Vector column
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
df_assembled = assembler.transform(df_model).select("FLIGHT_ID", "features", "label")

# Split Data (70% train / 30% test)
train_data, test_data = df_assembled.randomSplit([0.7, 0.3], seed=42)

# Initialize Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=5, seed=42)

print("Training Spark ML Random Forest...")
# Fit model
dt_model = rf.fit(train_data)

# Predict
predictions = dt_model.transform(test_data)

# Evaluate Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Test accuracy: {accuracy:.4f}")

Total Initial Samples: 44610 | Features: 72
Balanced Dataset Samples: 14960 (50% delayed, 50% on-time)
Training Spark ML Random Forest...


Test accuracy: 0.5977


In [7]:
# Extract Feature Importances matching them with their names
import pandas as pd

importances = dt_model.featureImportances.toArray()
feature_importances = sorted(zip(feature_cols, importances), key=lambda x: -x[1])

print("Top 10 features by importance:")
for feat, imp in feature_importances[:10]:
    print(f"{feat}: {imp:.4f}")

Top 10 features by importance:
ORG_Visibility_H2: 0.0530
ORG_Visibility_H4: 0.0487
ORG_Visibility_H0: 0.0380
ORG_Visibility_H8: 0.0376
DST_WindSpeed_H4: 0.0351
ORG_Visibility_H6: 0.0338
ORG_Visibility_H1: 0.0313
ORG_Visibility_H5: 0.0286
DST_Visibility_H2: 0.0272
DST_Visibility_H3: 0.0272


### Summary

- **Data preparation** (Section 1): PySpark for loading and robust 12-hour window lookbacks using bounded interval joins and Window ranking algorithms. Replaces Pandas `merge_asof`.
- **Weather features extraction**: Calculates historical lags simultaneously for `ORIGIN` and `DESTINATION`.
- **Prediction** (Section 2): Balanced classes through Random Under-Sampling prior to executing a Spark ML `VectorAssembler` and `RandomForestClassifier`. 

This Spark version cleanly mitigates RAM limits associated with `pd.merge_asof` and seamlessly builds an end-to-end framework deployable to PySpark clusters.